# BlueSky Modeling Results After 2026-03-19

This notebook loads the analysis artifacts generated for the post-2026-03-19 live archive and prints the main RQ1 results, sensitivity checks, gatekeeping split, and high-residual duplicate examples.


In [1]:
from __future__ import annotations

import csv
import json
import subprocess
from pathlib import Path
from typing import Iterable


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "Blue Sky Modeling.pdf").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from current working directory.")


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv_rows(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def print_section(title: str) -> None:
    print()
    print("=" * 88)
    print(title)
    print("=" * 88)


def print_kv(items: Iterable[tuple[str, object]]) -> None:
    for key, value in items:
        print(f"{key}: {value}")


repo_root = find_repo_root(Path.cwd())
analysis_dir = repo_root / "output" / "analysis_demo_20260323"
notebook_dir = repo_root / "output" / "jupyter-notebook"
study_root = repo_root / "data_v2_full" / "micro5" / "micro10_full_live_20260319" / "micro5_core_full"

artifacts = {
    "main": analysis_dir / "dced_trajectory_gap_metrics_micro10_full_live_post20260319_24h_1h_availability_strict20m_context_ever_seen.json",
    "content_same": analysis_dir / "dced_trajectory_gap_metrics_micro10_full_live_post20260319_24h_1h_availability_strict20m_context_ever_seen_content_same.json",
    "age_window": analysis_dir / "dced_trajectory_gap_metrics_micro10_full_live_post20260319_24h_1h_availability_strict20m_context_age_window.json",
    "gatekeeping": analysis_dir / "dced_gatekeeping_audit_micro10_full_live_post20260319_24h_1h_availability_strict20m_context_ever_seen_manual.json",
    "cluster_json": analysis_dir / "dced_cluster_purity_audit_micro10_full_live_post20260319_24h_1h_availability_strict20m_context_ever_seen.json",
    "cluster_csv": analysis_dir / "dced_cluster_purity_audit_micro10_full_live_post20260319_24h_1h_availability_strict20m_context_ever_seen.csv",
}

main_result = load_json(artifacts["main"])
content_same_result = load_json(artifacts["content_same"])
age_window_result = load_json(artifacts["age_window"])
gatekeeping_result = load_json(artifacts["gatekeeping"])
cluster_result = load_json(artifacts["cluster_json"])
cluster_rows = load_csv_rows(artifacts["cluster_csv"])

print(f"repo_root: {repo_root}")
print(f"analysis_dir: {analysis_dir}")
print(f"notebook_dir: {notebook_dir}")


repo_root: /Volumes/T9/BlueSky
analysis_dir: /Volumes/T9/BlueSky/output/analysis_demo_20260323
notebook_dir: /Volumes/T9/BlueSky/output/jupyter-notebook


## Optional: regenerate the analysis artifacts

The notebook is set up to *read* the already-generated JSON and CSV outputs by default. If you want to rerun the heavy analysis jobs from inside the notebook, switch `RUN_HEAVY_ANALYSIS` to `True`.


In [2]:
RUN_HEAVY_ANALYSIS = False

commands = [
    [
        "python3",
        "scripts/run_dced_trajectory_gap_metrics.py",
        "--root", str(repo_root / "data_v2_full"),
        "--study-id", "micro10_full_live_20260319",
        "--out-json", str(artifacts["main"]),
        "--max-age-hours", "24",
        "--riskset-mode", "ever_seen_in_feed",
        "--early-window-hours", "1",
        "--availability-anchor", "availability_time",
        "--max-first-monitor-delay-minutes", "20",
        "--strict-cohort-mode", "context",
    ],
    [
        "python3",
        "scripts/run_dced_trajectory_gap_metrics_content_same.py",
        "--root", str(repo_root / "data_v2_full"),
        "--study-id", "micro10_full_live_20260319",
        "--out-json", str(artifacts["content_same"]),
        "--signature-cache-json", str(analysis_dir / "content_signature_cache_micro10_full_live_20260319.json"),
        "--max-age-hours", "24",
        "--riskset-mode", "ever_seen_in_feed",
        "--early-window-hours", "1",
        "--availability-anchor", "availability_time",
        "--max-first-monitor-delay-minutes", "20",
        "--strict-cohort-mode", "context",
    ],
    [
        "python3",
        "scripts/run_dced_trajectory_gap_metrics.py",
        "--root", str(repo_root / "data_v2_full"),
        "--study-id", "micro10_full_live_20260319",
        "--out-json", str(artifacts["age_window"]),
        "--max-age-hours", "24",
        "--riskset-mode", "age_window",
        "--early-window-hours", "1",
        "--availability-anchor", "availability_time",
        "--max-first-monitor-delay-minutes", "20",
        "--strict-cohort-mode", "context",
    ],
]

if RUN_HEAVY_ANALYSIS:
    for cmd in commands:
        print("RUN:", " ".join(cmd))
        completed = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True, check=True)
        print(completed.stdout)
else:
    print("RUN_HEAVY_ANALYSIS is False. Using existing artifacts from output/analysis_demo_20260323.")


RUN_HEAVY_ANALYSIS is False. Using existing artifacts from output/analysis_demo_20260323.


## Archive coverage

This cell prints the filesystem coverage of the live study slice used by the analysis.


In [3]:
window_dirs = sorted(path for path in study_root.glob("*/*/*") if path.is_dir())
day_dirs = sorted(path for path in study_root.glob("*") if path.is_dir())
first_window = window_dirs[0] if window_dirs else None
last_window = window_dirs[-1] if window_dirs else None

def window_dir_to_utc(path: Path, *, end: bool = False) -> str:
    day = path.parent.parent.name
    hour = path.parent.name
    minute = path.name
    suffix = ":10:00Z" if end else ":00:00Z"
    return f"{day}T{hour}:{minute}{suffix}"

print_section("Archive coverage")
print_kv([
    ("study_root", study_root),
    ("day_count", len(day_dirs)),
    ("days", [path.name for path in day_dirs]),
    ("window_dir_count", len(window_dirs)),
    ("first_window_start_utc", window_dir_to_utc(first_window, end=False) if first_window else None),
    ("last_window_end_utc", window_dir_to_utc(last_window, end=True) if last_window else None),
    ("feed_posts_in_main_result", main_result["feed_posts"]),
])



Archive coverage
study_root: /Volumes/T9/BlueSky/data_v2_full/micro5/micro10_full_live_20260319/micro5_core_full
day_count: 5
days: ['2026-03-19', '2026-03-20', '2026-03-21', '2026-03-22', '2026-03-23']
window_dir_count: 599
first_window_start_utc: 2026-03-19T05:30:00:00Z
last_window_end_utc: 2026-03-23T17:00:10:00Z
feed_posts_in_main_result: 624430


## Main RQ1 result

The main specification is the feed-aware `ever_seen_in_feed` risk set with `availability_time`, a `20m` strict cohort, and `1h` early trajectory features.


In [4]:
gap = main_result["gap_summary"]
cohort = main_result["strict_cohort_summary"]

print_section("RQ1 main result: ever_seen_in_feed")
print_kv([
    ("riskset_definition", main_result["riskset_definition"]),
    ("valid_duplicate_clusters", main_result["valid_duplicate_clusters"]),
    ("riskset_contexts_ge2", main_result["riskset_contexts_ge2"]),
    ("riskset_rows", main_result["riskset_rows"]),
    ("riskset_zero_exposure_share", main_result["riskset_zero_exposure_share"]),
    ("contexts_after_filter", cohort["contexts_after_filter"]),
    ("exposure_after_filter", cohort["exposure_after_filter"]),
    ("weighted_mean_gap_equal", gap["weighted_mean_gap_equal"]),
    ("weighted_mean_gap_timing", gap["weighted_mean_gap_timing"]),
    ("weighted_mean_gap_timing_author_trajectory", gap["weighted_mean_gap_timing_author_trajectory"]),
    ("timing_explained_share", gap["timing_explained_share"]),
    ("final_unexplained_share", gap["final_unexplained_share"]),
])

print()
print("Main model coefficients:")
for model_name, params in main_result["model_params"].items():
    print(f"  {model_name}")
    for key, value in params.items():
        print(f"    {key}: {value}")



RQ1 main result: ever_seen_in_feed
riskset_definition: same duplicate cluster, same feed/viewer/window, eligible if created_at <= window_end and post was ever observed in the same feed
valid_duplicate_clusters: 4399
riskset_contexts_ge2: 1929
riskset_rows: 4401
riskset_zero_exposure_share: 0.2509
contexts_after_filter: 1929
exposure_after_filter: 948.672403
weighted_mean_gap_equal: 0.1699
weighted_mean_gap_timing: 0.0551
weighted_mean_gap_timing_author_trajectory: 0.058
timing_explained_share: 0.6758
final_unexplained_share: 0.3414

Main model coefficients:
  timing
    age_spline_0: -0.0
    age_spline_1: -7.216165
    age_spline_2: 6.753253
    age_spline_3: -67.115153
  timing_author
    age_spline_0: -0.0
    age_spline_1: -7.201256
    age_spline_2: 6.672854
    age_spline_3: -67.051947
    log10_followers: 0.079954
    log10_posts: -0.07642
  timing_trajectory
    age_spline_0: -0.0
    age_spline_1: -6.950804
    age_spline_2: 6.277209
    age_spline_3: -66.35961
    early_prio

## Sensitivity checks

This cell compares the main result against the stricter `content-same` duplicate definition and the looser `age_window` risk set.


In [5]:
comparison_rows = [
    ("main_ever_seen", main_result),
    ("strict_content_same", content_same_result),
    ("age_window", age_window_result),
]

print_section("Sensitivity comparison")
header = [
    "label",
    "contexts",
    "rows",
    "gap_equal",
    "gap_timing",
    "gap_final",
    "timing_explained",
    "final_unexplained",
]
print(" | ".join(header))
print("-" * 120)
for label, payload in comparison_rows:
    gap = payload["gap_summary"]
    print(" | ".join([
        label,
        str(payload["riskset_contexts_ge2"]),
        str(payload["riskset_rows"]),
        str(gap["weighted_mean_gap_equal"]),
        str(gap["weighted_mean_gap_timing"]),
        str(gap["weighted_mean_gap_timing_author_trajectory"]),
        str(gap["timing_explained_share"]),
        str(gap["final_unexplained_share"]),
    ]))

print()
print("Content-same live signature coverage:")
for key, value in content_same_result["live_signature_summary"].items():
    print(f"  {key}: {value}")



Sensitivity comparison
label | contexts | rows | gap_equal | gap_timing | gap_final | timing_explained | final_unexplained
------------------------------------------------------------------------------------------------------------------------
main_ever_seen | 1929 | 4401 | 0.1699 | 0.0551 | 0.058 | 0.6758 | 0.3414
strict_content_same | 692 | 1530 | 0.1288 | 0.0506 | 0.0512 | 0.6072 | 0.3974
age_window | 2313 | 5674 | 0.216 | 0.1559 | 0.1465 | 0.2785 | 0.6782

Content-same live signature coverage:
  candidate_posts_for_live_signature: 14508
  fetched_signature_posts: 14184
  missing_signature_posts: 324
  signature_coverage_share: 0.9777
  signature_cache_path: /Volumes/T9/BlueSky/output/analysis_demo_20260323/content_signature_cache_micro10_full_live_20260319.json


## Gatekeeping vs in-ranking

The wrapper script in the repo had a stale function signature, so the gatekeeping artifact used here was recomputed directly from the underlying model functions.


In [6]:
print_section("Gatekeeping decomposition")
overall = gatekeeping_result["summary"]["overall"]
print_kv([(key, value) for key, value in overall.items()])

print_section("By viewer")
for row in gatekeeping_result["summary"]["by_viewer"]:
    print_kv([(key, value) for key, value in row.items()])
    print("-" * 60)

print_section("By bucket")
for row in gatekeeping_result["summary"]["by_bucket"]:
    print_kv([(key, value) for key, value in row.items()])
    print("-" * 60)



Gatekeeping decomposition
label: overall
n: 1933
weighted_zero_share: 0.1311
raw_total: 0.17
raw_gatekeeping: 0.0655
raw_in_ranking: 0.1044
raw_gatekeeping_share: 0.3856
timing_total: 0.055
timing_gatekeeping: 0.0104
timing_in_ranking: 0.0446
timing_gatekeeping_share: 0.1897
final_total: 0.0579
final_gatekeeping: 0.0101
final_in_ranking: 0.0479
final_gatekeeping_share: 0.1737
weight_share: 1.0

By viewer
label: viewer=auth
n: 957
weighted_zero_share: 0.131
raw_total: 0.1665
raw_gatekeeping: 0.0655
raw_in_ranking: 0.1011
raw_gatekeeping_share: 0.3932
timing_total: 0.0535
timing_gatekeeping: 0.0103
timing_in_ranking: 0.0431
timing_gatekeeping_share: 0.1931
final_total: 0.057
final_gatekeeping: 0.01
final_in_ranking: 0.0471
final_gatekeeping_share: 0.1747
weight_share: 0.4939
------------------------------------------------------------
label: viewer=unauth
n: 976
weighted_zero_share: 0.1312
raw_total: 0.1733
raw_gatekeeping: 0.0656
raw_in_ranking: 0.1077
raw_gatekeeping_share: 0.3785
tim

## Top residual duplicate clusters

These are the most uneven duplicate clusters after the final timing+author+trajectory adjustment under the main specification.


In [7]:
print_section("Top residual clusters summary")
for rank, row in enumerate(cluster_result["top_clusters_by_final_residual"][:10], start=1):
    print_kv([
        ("rank", rank),
        ("cluster_text", row["cluster_text"][:180]),
        ("weighted_mean_gap_equal", row["weighted_mean_gap_equal"]),
        ("weighted_mean_gap_timing_author_trajectory", row["weighted_mean_gap_timing_author_trajectory"]),
        ("cluster_total_context_exposure", row["cluster_total_context_exposure"]),
        ("mean_riskset_size", row["mean_riskset_size"]),
        ("mean_shown_count", row["mean_shown_count"]),
    ])
    print("-" * 80)

print_section("Sample rows from the cluster audit CSV")
for row in cluster_rows[:12]:
    keep = {
        key: row[key]
        for key in [
            "cluster_rank",
            "cluster_text",
            "post_uri",
            "author_handle",
            "record_created_at",
            "first_monitored_delay_minutes",
            "total_exposure",
            "shown_context_count",
        ]
        if key in row
    }
    print(keep)



Top residual clusters summary
rank: 1
cluster_text: the bruv is back! will ospreay returned to action against blake christian, but immediately shifted gears to jon moxley + the death riders! watch the #aewdynamite replay right now o
weighted_mean_gap_equal: 0.5
weighted_mean_gap_timing_author_trajectory: 0.558722
cluster_total_context_exposure: 0.787318
mean_riskset_size: 2.0
mean_shown_count: 1.0
--------------------------------------------------------------------------------
rank: 2
cluster_text: trump accidentally reveals what he really thinks of maga voters newrepublic.com/article/2078... via @newrepublic.com
weighted_mean_gap_equal: 0.666667
weighted_mean_gap_timing_author_trajectory: 0.525262
cluster_total_context_exposure: 0.501616
mean_riskset_size: 3.0
mean_shown_count: 1.0
--------------------------------------------------------------------------------
rank: 3
cluster_text: pon un gif de tu pokémon favorito.
weighted_mean_gap_equal: 0.666667
weighted_mean_gap_timing_author_t

## RQ2 boundary check

The current post-2026-03-19 archive does not yet have the event/frame labels required for the RQ2 frame-disparity formulas in the PDF.


In [8]:
label_files = sorted((repo_root / "_build").glob("**/*cluster_labels*.csv"))
recent_label_files = [path for path in label_files if "2026-03-19" in str(path) or "2026-03-20" in str(path) or "2026-03-21" in str(path) or "2026-03-22" in str(path) or "2026-03-23" in str(path)]
demo_labeled = repo_root / "_build" / "2026-03-13" / "llm_annotation_demo_500_v2_applied_partial" / "annotation_demo_labeled_examples.csv"

print_section("RQ2 label availability")
print_kv([
    ("cluster_label_file_count_total", len(label_files)),
    ("cluster_label_file_count_on_or_after_2026_03_19", len(recent_label_files)),
    ("demo_labeled_examples_exists", demo_labeled.exists()),
    ("demo_labeled_examples_path", demo_labeled),
])

print()
print("Sample label files found:")
for path in label_files[:10]:
    print(f"  {path}")

if not recent_label_files:
    print()
    print("No frame-label files were found for the post-2026-03-19 live archive. RQ2 is not yet computable from the current archive without a new labeling pass.")



RQ2 label availability
cluster_label_file_count_total: 6
cluster_label_file_count_on_or_after_2026_03_19: 0
demo_labeled_examples_exists: True
demo_labeled_examples_path: /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2_applied_partial/annotation_demo_labeled_examples.csv

Sample label files found:
  /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2/._climate_cluster_labels.csv
  /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2/._gaza_cluster_labels_partial.csv
  /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2/._immigration_cluster_labels_partial.csv
  /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2/climate_cluster_labels.csv
  /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2/gaza_cluster_labels_partial.csv
  /Volumes/T9/BlueSky/_build/2026-03-13/llm_annotation_demo_500_v2/immigration_cluster_labels_partial.csv

No frame-label files were found for the post-2026-03-19 live archive. RQ2 is n

## Final takeaway

This final cell prints the compact conclusion that matches the analysis summary.


In [9]:
final_takeaway = {
    "main_spec": {
        "raw_gap": main_result["gap_summary"]["weighted_mean_gap_equal"],
        "timing_gap": main_result["gap_summary"]["weighted_mean_gap_timing"],
        "final_gap": main_result["gap_summary"]["weighted_mean_gap_timing_author_trajectory"],
        "timing_explained_share": main_result["gap_summary"]["timing_explained_share"],
        "final_unexplained_share": main_result["gap_summary"]["final_unexplained_share"],
    },
    "strict_content_same": {
        "raw_gap": content_same_result["gap_summary"]["weighted_mean_gap_equal"],
        "final_gap": content_same_result["gap_summary"]["weighted_mean_gap_timing_author_trajectory"],
        "final_unexplained_share": content_same_result["gap_summary"]["final_unexplained_share"],
    },
    "age_window_sensitivity": {
        "raw_gap": age_window_result["gap_summary"]["weighted_mean_gap_equal"],
        "final_gap": age_window_result["gap_summary"]["weighted_mean_gap_timing_author_trajectory"],
        "final_unexplained_share": age_window_result["gap_summary"]["final_unexplained_share"],
    },
    "gatekeeping_overall": {
        "raw_gatekeeping_share": gatekeeping_result["summary"]["overall"]["raw_gatekeeping_share"],
        "timing_gatekeeping_share": gatekeeping_result["summary"]["overall"]["timing_gatekeeping_share"],
        "final_gatekeeping_share": gatekeeping_result["summary"]["overall"]["final_gatekeeping_share"],
    },
    "rq2_ready": False,
}

print_section("Compact takeaway")
print(json.dumps(final_takeaway, indent=2))



Compact takeaway
{
  "main_spec": {
    "raw_gap": 0.1699,
    "timing_gap": 0.0551,
    "final_gap": 0.058,
    "timing_explained_share": 0.6758,
    "final_unexplained_share": 0.3414
  },
  "strict_content_same": {
    "raw_gap": 0.1288,
    "final_gap": 0.0512,
    "final_unexplained_share": 0.3974
  },
  "age_window_sensitivity": {
    "raw_gap": 0.216,
    "final_gap": 0.1465,
    "final_unexplained_share": 0.6782
  },
  "gatekeeping_overall": {
    "raw_gatekeeping_share": 0.3856,
    "timing_gatekeeping_share": 0.1897,
    "final_gatekeeping_share": 0.1737
  },
  "rq2_ready": false
}
